In [ ]:

import math
import pathlib
import time

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from refig import refig

from zfish.invariants import compute_invariants, select_invariants

plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["font.family"] = "Roboto"


In [ ]:

CONTACTOME_EDGELIST_PATH = pathlib.Path("../data/graphs/contactomes/")
# THRESH = 1_000_000 # one square micron
THRESH = 100_000  # one half square micron


In [ ]:

def load_contactome_csv_to_graph(csv_path: pathlib.Path) -> nx.Graph:
    graph = nx.Graph()
    with open(csv_path, "r") as f:
        next(f)  # Skip header
        for line in f:
            clean = line.strip()
            if not clean:
                continue
            pre, post, weight = clean.split(",")
            weight_value = int(float(weight))
            if graph.has_edge(pre, post):
                graph[pre][post]["weight"] += weight_value
            else:
                graph.add_edge(pre, post, weight=weight_value)
    return graph


def load_all_graphs_from_edgelists(edgelist_path: pathlib.Path) -> dict[str, nx.Graph]:
    graphs = {}
    for edgelist_file in edgelist_path.glob("*825.csv"):
        simplified_file = edgelist_file.with_name(
            edgelist_file.stem.replace("-100825", f"-simple.thresh{THRESH}")
            + edgelist_file.suffix
        )
        if simplified_file.exists():
            print(f"Loading simplified graph from {simplified_file}")
            graph = load_contactome_csv_to_graph(simplified_file)
        else:
            print(f"Loading graph from {edgelist_file}")
            graph = load_contactome_csv_to_graph(edgelist_file)

        graph_name = edgelist_file.stem
        identifier = graph_name.split("-")[1]
        graphs[identifier] = graph
    return dict(sorted(graphs.items(), key=lambda item: int(item[0][:-1])))


In [ ]:

raw_graphs = load_all_graphs_from_edgelists(CONTACTOME_EDGELIST_PATH)


In [ ]:

# UNCOMMENT THESE LINES TO SAVE OFF NEW FILES:
# Dump the graphs to new .simple.thresh##.csv files.
for identifier, graph in raw_graphs.items():
    if THRESH is not None:
        output_path = CONTACTOME_EDGELIST_PATH / f"contactome-{identifier}-simple.thresh{THRESH}.csv"
    else:
        output_path = CONTACTOME_EDGELIST_PATH / f"contactome-{identifier}-simple.csv"
    with open(output_path, "w") as f:
        f.write("pre,post,weight
")
        for pre, post, data in graph.edges(data=True):
            weight = data["weight"]
            if THRESH is not None and weight < THRESH:
                continue
            f.write(f"{pre},{post},{weight}
")
# Load the latest:
raw_graphs = load_all_graphs_from_edgelists(CONTACTOME_EDGELIST_PATH)
print("Done dumping simple graphs.")


In [ ]:

graphs = {}
for identifier, graph in raw_graphs.items():
    print("Processing graph:", identifier)
    if graph.number_of_nodes() > 0:
        largest_component_nodes = max(nx.connected_components(graph), key=len)
    else:
        largest_component_nodes = []
    lcc_graph = (
        graph.subgraph(largest_component_nodes).copy()
        if largest_component_nodes
        else graph.copy()
    )
    graphs[identifier] = {
        "graph": graph,
        "undirected_graph": graph,
        "lcc_graph": lcc_graph,
    }


In [ ]:

INVARIANTS = select_invariants("contactome")

invariants_per_graph = compute_invariants(graphs, "contactome", INVARIANTS)


In [ ]:

with plt.style.context("bmh"):
    fig, axs = plt.subplots(
        len(INVARIANTS), 1, figsize=(3, 2 * len(INVARIANTS)), dpi=150, sharex=True
    )
    for ax, (invariant_key, invariant_spec) in zip(axs, INVARIANTS.items()):
        ylabel = invariant_spec.ylabel
        graph_ids = list(invariants_per_graph[invariant_key].keys())
        values = list(invariants_per_graph[invariant_key].values())
        ax.plot(graph_ids, values, marker="o")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{ylabel}")
        ax.grid(True)
    plt.tight_layout()
    ax.set_xlabel("Graph Identifier")
    refig.savefig("contactome_invariants_per_graph.svg")
    plt.show()


In [ ]:

with plt.style.context("seaborn-v0_8-poster"):
    fig, axs = plt.subplots(
        len(INVARIANTS), 1, figsize=(6, 4 * len(INVARIANTS)), dpi=150, sharex=True
    )
    for ax, (invariant_key, invariant_spec) in zip(axs, INVARIANTS.items()):
        ylabel = invariant_spec.ylabel
        graph_ids = list(invariants_per_graph[invariant_key].keys())
        values = np.array(list(invariants_per_graph[invariant_key].values()))
        zscores = (values - np.mean(values)) / np.std(values)
        ax.plot(graph_ids, zscores, marker="o")
        ax.set_ylabel(f"Z-score of {ylabel}")
        ax.set_title(f"Z-score of {ylabel}")
        ax.grid(True)
    plt.tight_layout()
    ax.set_xlabel("Graph Identifier")
    refig.savefig("contactome_invariants_per_graph_zscores.svg")
    plt.show()


In [ ]:

with plt.style.context("seaborn-v0_8-poster"):
    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    for invariant_key, invariant_spec in INVARIANTS.items():
        ylabel = invariant_spec.ylabel
        graph_ids = list(invariants_per_graph[invariant_key].keys())
        values = np.array(list(invariants_per_graph[invariant_key].values()))
        zscores = (values - np.mean(values)) / np.std(values)
        ax.plot(graph_ids, zscores, marker="o", label=ylabel)
    ax.set_ylabel("Z-score of Invariants")
    ax.set_title("Z-scores of Invariants Across Graphs")
    ax.grid(True)
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:

with open("contactome_invariants_per_graph.json", "w") as f:
    import json

    json.dump(invariants_per_graph, f, indent=2)
